# PLXG Novel - Qwen vLLM API on Kaggle

Notebook nay khoi dong LLM dich truyen tren Kaggle va expose API OpenAI-compatible cho backend. Endpoint khop voi `HF_ENDPOINT`, bearer token khop voi `HF_TOKEN`, khong can doi frontend hay request DTO.

Truoc khi chay: bat `Internet` va `GPU` trong Kaggle Notebook Settings. Dat secret `NGROK_AUTHTOKEN` va `VLLM_API_KEY` trong Add-ons -> Secrets, sau do cap quyen truy cap cho notebook nay. `VLLM_API_KEY` phai la mot chuoi ngau nhien dai va se dung lam `HF_TOKEN` o backend.

In [ ]:
import os
import subprocess
from pathlib import Path

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OMP_NUM_THREADS'] = str(min(8, os.cpu_count() or 4))

subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], check=True)

In [ ]:
import sys
import shutil
import importlib.util

clean_env = os.environ.copy()
clean_env.pop('PYTHONPATH', None)
clean_env.pop('PYTHONHOME', None)

BOOTSTRAP_DIR = Path('/kaggle/working/bootstrap_tools')
BOOTSTRAP_DIR.mkdir(parents=True, exist_ok=True)
if importlib.util.find_spec('virtualenv') is None:
    subprocess.run([
        sys.executable,
        '-I',
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--target',
        str(BOOTSTRAP_DIR),
        'virtualenv>=20.26.0,<21.0.0',
    ], check=True, env=clean_env)

if str(BOOTSTRAP_DIR) not in sys.path:
    sys.path.insert(0, str(BOOTSTRAP_DIR))

import virtualenv

VLLM_VENV_DIR = Path('/kaggle/working/vllm_venv')
VENV_BIN_DIR = VLLM_VENV_DIR / 'bin'
PYTHON_BIN = str(VENV_BIN_DIR / 'python')
PIP_BIN = str(VENV_BIN_DIR / 'pip')
MARKER_FILE = VLLM_VENV_DIR / '.plxg_vllm_ready'

rebuild_env = not Path(PYTHON_BIN).exists() or not Path(PIP_BIN).exists() or not MARKER_FILE.exists()
if rebuild_env:
    shutil.rmtree('/kaggle/working/vllm_site', ignore_errors=True)
    shutil.rmtree(VLLM_VENV_DIR, ignore_errors=True)
    virtualenv.cli_run([str(VLLM_VENV_DIR)])
    subprocess.run([
        PIP_BIN,
        'install',
        '-q',
        '--upgrade',
        'pip',
        'setuptools',
        'wheel',
        'vllm>=0.9.0,<0.10.0',
        'transformers==4.51.3',
        'pyngrok>=7.2.8,<8.0.0',
        'httpx>=0.28.1,<0.29.0',
    ], check=True, env=clean_env)
    MARKER_FILE.write_text('ready')
else:
    print('Reuse existing vLLM environment')

venv_site_packages = VLLM_VENV_DIR / 'lib' / f"python{sys.version_info.major}.{sys.version_info.minor}" / 'site-packages'
if str(venv_site_packages) not in sys.path:
    sys.path.insert(0, str(venv_site_packages))

existing_pythonpath = os.environ.get('PYTHONPATH')
os.environ['PYTHONPATH'] = str(venv_site_packages) if not existing_pythonpath else f"{venv_site_packages}:{existing_pythonpath}"

print(f'Isolated venv: {VLLM_VENV_DIR}')
print(f'Site-packages: {venv_site_packages}')
print(f'Rebuilt environment: {rebuild_env}')
print(f'Python binary: {PYTHON_BIN}')

## Ghi chu dependency

Cell cai dat dung `virtualenv` bootstrap trong `/kaggle/working/bootstrap_tools` de tao moi truong rieng `/kaggle/working/vllm_venv` ma khong phu thuoc `ensurepip` cua runtime Kaggle. Cac subprocess Python duoc chay voi `-I` va `clean_env` de tranh `sitecustomize`/`PYTHONPATH` ban cua Kaggle. De tang toc do, cell nay se tai su dung `vllm_venv` neu da co marker `.plxg_vllm_ready`; chi khi env thieu file hoac chua setup xong moi rebuild va cai lai `vLLM`. `transformers` duoc pin ve `4.51.3` de tranh loi `ValueError: aimv2 is already used by a Transformers config`. Neu cong chua tung chay ban cu `pip install -U ...` tren global env va thay warning `protobuf`, `requests`, `starlette`, `numba`, cach sach nhat la `Restart session` roi chay lai notebook nay tu dau.

In [ ]:
import json
import re
from kaggle_secrets import UserSecretsClient

PORT = 8000
MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.9
MAX_NUM_SEQS = 6
BENCHMARK_CONCURRENCY = 1

secrets = UserSecretsClient()
NGROK_AUTHTOKEN = secrets.get_secret('NGROK_AUTHTOKEN')
VLLM_API_KEY = secrets.get_secret('VLLM_API_KEY')

if not NGROK_AUTHTOKEN or not VLLM_API_KEY:
    raise RuntimeError('Can tao va cap quyen secrets NGROK_AUTHTOKEN, VLLM_API_KEY trong Kaggle.')

gpu_query = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    text=True,
).strip().splitlines()
gpu_memory_mib = [int(re.search(r'(\d+)', line.split(',')[-1]).group(1)) for line in gpu_query]
gpu_count = len(gpu_memory_mib)
smallest_gpu_memory_mib = min(gpu_memory_mib)

if gpu_count >= 2 and smallest_gpu_memory_mib >= 14000:
    MODEL = 'Qwen/Qwen2.5-14B-Instruct-AWQ'
    QUANTIZATION = 'awq'
    TENSOR_PARALLEL_SIZE = 2
    DISABLE_CUSTOM_ALL_REDUCE = True
    ENFORCE_EAGER = True
else:
    MODEL = 'Qwen/Qwen2.5-7B-Instruct'
    QUANTIZATION = None
    TENSOR_PARALLEL_SIZE = 1
    DISABLE_CUSTOM_ALL_REDUCE = False
    ENFORCE_EAGER = False

print(json.dumps({
    'gpus': gpu_query,
    'model': MODEL,
    'quantization': QUANTIZATION,
    'tensor_parallel_size': TENSOR_PARALLEL_SIZE,
    'disable_custom_all_reduce': DISABLE_CUSTOM_ALL_REDUCE,
    'enforce_eager': ENFORCE_EAGER,
    'max_model_len': MAX_MODEL_LEN,
    'benchmark_concurrency': BENCHMARK_CONCURRENCY,
}, indent=2))

In [ ]:
LOG_PATH = Path('/kaggle/working/vllm.log')
command = [
    PYTHON_BIN, '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--host', '0.0.0.0',
    '--port', str(PORT),
    '--api-key', VLLM_API_KEY,
    '--tensor-parallel-size', str(TENSOR_PARALLEL_SIZE),
    '--gpu-memory-utilization', str(GPU_MEMORY_UTILIZATION),
    '--max-model-len', str(MAX_MODEL_LEN),
    '--max-num-seqs', str(MAX_NUM_SEQS),
    '--enable-prefix-caching',
    '--disable-log-requests',
]

if QUANTIZATION:
    command.extend(['--quantization', QUANTIZATION])
if DISABLE_CUSTOM_ALL_REDUCE:
    command.append('--disable-custom-all-reduce')
if ENFORCE_EAGER:
    command.append('--enforce-eager')

with LOG_PATH.open('w') as log_file:
    server_process = subprocess.Popen(command, stdout=log_file, stderr=subprocess.STDOUT, env=os.environ.copy())

print(f'vLLM PID: {server_process.pid}')
print(' '.join(command))

In [ ]:
import time
import requests

health_url = f'http://127.0.0.1:{PORT}/health'
deadline = time.monotonic() + 900

while time.monotonic() < deadline:
    if server_process.poll() is not None:
        raise RuntimeError(LOG_PATH.read_text()[-8000:])
    try:
        response = requests.get(health_url, timeout=5)
        if response.ok:
            print('vLLM da san sang')
            break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    raise TimeoutError(LOG_PATH.read_text()[-8000:])

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
tunnel = ngrok.connect(PORT, proto='http')
PUBLIC_BASE_URL = tunnel.public_url
LOCAL_COMPLETIONS_ENDPOINT = f'http://127.0.0.1:{PORT}/v1/chat/completions'
HF_ENDPOINT = f'{PUBLIC_BASE_URL}/v1/chat/completions'

print('Dat cac bien nay trong backend/.env:')
print(f'HF_ENDPOINT={HF_ENDPOINT}')
print('HF_TOKEN=<gia tri secret VLLM_API_KEY>')
print(f'HF_MODEL={MODEL}')
print('TRANSLATION_MAX_IN_FLIGHT=2')
print(f'Local benchmark endpoint: {LOCAL_COMPLETIONS_ENDPOINT}')

In [ ]:
import asyncio
import httpx

SYSTEM_PROMPT = 'Ban la dich gia tieu thuyet mang Trung sang Viet. Nhan dien van phong goc de dung xung ho tu nhien: hien dai dung khau ngu, co dai dung xung ho co trang, tien hiep dung thuat ngu chuan. Dich tu nhien theo ngu phap Viet, giu day du y, khong them bot va khong dich tung chu. Dung glossary cho ten rieng, ten moi phai nhat quan. contextBefore chi de hieu mach, khong dich lai. Chi tra JSON hop le theo schema {\"paragraphs\":[{\"id\":\"...\",\"text\":\"...\"}]}. So luong va thu tu paragraphs phai khop input.'

def build_payload(paragraph_id):
    return {
        'model': MODEL,
        'temperature': 0.2,
        'max_tokens': 128,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {
                'role': 'user',
                'content': json.dumps({
                    'contextBefore': '',
                    'glossary': [],
                    'paragraphs': [{'id': paragraph_id, 'text': '她轻声说道，今天风很冷。'}],
                }, ensure_ascii=False),
            },
        ],
    }

async def request_translation(client, payload, endpoint):
    response = await client.post(
        endpoint,
        headers={'Authorization': f'Bearer {VLLM_API_KEY}'},
        json=payload,
    )
    if response.status_code >= 400:
        raise RuntimeError(f'HTTP {response.status_code}: {response.text[:4000]}')
    content = response.json()['choices'][0]['message']['content']
    try:
        parsed = json.loads(content)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f'Provider did not return valid JSON: {exc}; raw={content[:4000]}')
    paragraphs = parsed.get('paragraphs')
    if not isinstance(paragraphs, list) or not paragraphs:
        raise RuntimeError(f'Provider JSON missing paragraphs: {content[:4000]}')
    return content

async def run_parallel_benchmark(concurrency):
    limits = httpx.Limits(max_connections=concurrency, max_keepalive_connections=concurrency)
    timeout = httpx.Timeout(120.0, connect=20.0)
    async with httpx.AsyncClient(limits=limits, timeout=timeout) as client:
        warmup_result = await request_translation(client, build_payload('warmup'), LOCAL_COMPLETIONS_ENDPOINT)
        started_at = time.perf_counter()
        results = await asyncio.gather(*(
            request_translation(client, build_payload(f'p{index:04d}'), LOCAL_COMPLETIONS_ENDPOINT)
            for index in range(concurrency)
        ))
    return warmup_result, results, time.perf_counter() - started_at

async def find_stable_concurrency():
    warmup_result = None
    best_results = []
    best_elapsed = 0.0
    stable_concurrency = 0
    for concurrency in range(1, BENCHMARK_CONCURRENCY + 1):
        try:
            warmup_result, results, elapsed_seconds = await run_parallel_benchmark(concurrency)
            stable_concurrency = concurrency
            best_results = results
            best_elapsed = elapsed_seconds
            print(f'Benchmark OK at concurrency={concurrency}: {elapsed_seconds:.2f}s')
        except Exception as exc:
            print(f'Benchmark failed at concurrency={concurrency}: {exc}')
            print(LOG_PATH.read_text()[-6000:])
            break
    if stable_concurrency == 0:
        raise RuntimeError('Khong qua duoc benchmark concurrency=1. Xem log ben tren.')
    return warmup_result, best_results, best_elapsed, stable_concurrency

try:
    warmup_result, benchmark_results, elapsed_seconds, stable_concurrency = await find_stable_concurrency()
except Exception as exc:
    print(f'Benchmark failed: {exc}')
    print(LOG_PATH.read_text()[-6000:])
    raise

print(warmup_result)
print(f'{len(benchmark_results)} requests song song trong {elapsed_seconds:.2f}s')
print(f'Stable benchmark concurrency: {stable_concurrency}')

metrics_response = requests.get(f'http://127.0.0.1:{PORT}/metrics', timeout=10)
metrics_response.raise_for_status()
prefix_cache_metrics = [
    line for line in metrics_response.text.splitlines()
    if 'prefix_cache_hit_rate' in line and not line.startswith('#')
]
print('\n'.join(prefix_cache_metrics) or 'Chua co metric prefix cache de hien thi.')

## Giu session

Chay cell heartbeat ben duoi neu muon giu notebook tiep tuc song va theo doi `vLLM` trong thoi gian dai.

In [ ]:
import datetime

HEARTBEAT_INTERVAL = 60
MAX_RUNTIME_HOURS = 11.5

started_at = time.monotonic()
print(f'Endpoint dang phuc vu: {HF_ENDPOINT}')
print(f'Bat dau giu session luc: {datetime.datetime.utcnow().isoformat()}Z')

while True:
    elapsed_hours = (time.monotonic() - started_at) / 3600
    if elapsed_hours >= MAX_RUNTIME_HOURS:
        print(f'Da chay {elapsed_hours:.2f}h, chu dong dung truoc gioi han 12h.')
        break

    if server_process.poll() is not None:
        print('vLLM process da thoat bat thuong. Log gan nhat:')
        print(LOG_PATH.read_text()[-4000:])
        break

    try:
        health = requests.get(health_url, timeout=5)
        status = 'OK' if health.ok else f'HTTP {health.status_code}'
    except requests.RequestException as exc:
        status = f'loi: {exc}'

    print(f'[{datetime.datetime.utcnow().isoformat()}Z] heartbeat - vLLM: {status}, da chay {elapsed_hours:.2f}h')
    time.sleep(HEARTBEAT_INTERVAL)


## Van hanh

Giu notebook nay dang chay khi backend gui job dich. Kaggle co the ket thuc session bat ky luc nao; khi khoi dong lai, lay `HF_ENDPOINT` moi va cap nhat bien moi truong cua backend.

Notebook warm-up mot request roi gui 4 request song song de kiem tra continuous batching. Khi metric `prefix_cache_hit_rate` tang, system prompt co dinh dang duoc cache. Tang `BENCHMARK_CONCURRENCY` va `TRANSLATION_MAX_IN_FLIGHT` tu 4 len toi da 8 tung buoc, chi khi GPU van con du VRAM va latency khong tang manh.

Neu gap OOM, doi `MAX_MODEL_LEN` thanh `4096`, `GPU_MEMORY_UTILIZATION` thanh `0.88`, roi chay lai tu cell cau hinh. Khong can sua request DTO hay endpoint backend.

Tai lieu chi tiet: `docs/translation/gpu-ai-setup.md`.